# Project 1- Notebook 1: Region 3 Performance Landscape
## 1. Setup

**Objective:** Load and explore the Region 3 fault ticket dataset

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Navigate to project root (2 levels up from notebooks/project1_national/)
os.chdir(os.path.join('..', '..'))
project_root = os.path.abspath(os.getcwd())

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("✅ Setup complete!")

# Plotting style (minimal, optional)
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

# Pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

✅ Setup complete!


## 2. Load Raw Data

In [2]:
from loading import load_data

# Load raw dataset
df_raw = load_data('data/raw/fault_tickets/national_dataset.csv')

# Basic overview
print(f"📊 Raw Data Overview:")
print(f"  Rows: {len(df_raw):,}")
print(f"  Columns: {len(df_raw.columns)}")
print("\n📋 Columns:")
print(df_raw.columns.tolist())

df_raw.head()

Falling back to synthetic dataset: data/raw/fault_tickets/synthetic_dataset.csv


📊 Raw Data Overview:
  Rows: 38,319
  Columns: 27

📋 Columns:
['TICKETID', 'Region', 'ZONE', 'CITY', 'Area', 'FT Status', 'Priority', 'Urgency', 'OUTAGEDURATION', 'RESOLUTION_TIME_HOURS', 'Standardized RFO', 'RFODescription', 'RootCause', 'Fault Type', 'StartDateTime', 'ActionTaken', 'NEType', 'SLA_Compliant', 'FT_Owner', 'WOLeadName', 'WOOwnerGroup', 'FIELD_TIME_HOURS', 'DISPATCH_DELAY_HOURS', 'Priority_Urgency', 'REPORTDATE', 'ContactGroup', 'DESCRIPTION']


,TICKETID,Region,ZONE,CITY,Area,FT Status,Priority,Urgency,OUTAGEDURATION,RESOLUTION_TIME_HOURS,Standardized RFO,RFODescription,RootCause,Fault Type,StartDateTime,ActionTaken,NEType,SLA_Compliant,FT_Owner,WOLeadName,WOOwnerGroup,FIELD_TIME_HOURS,DISPATCH_DELAY_HOURS,Priority_Urgency,REPORTDATE,ContactGroup,DESCRIPTION
0,TKT00035651,Region 3,ZONE 6,LAS PINAS CITY,Z6LAS0034,RESOLVED,3.0,3,83.5375,83.5375,FOC CUT WITH REDUNDANCY,FOC CUT WITH REDUNDANCY,Resolved – FOC CUT WITH REDUNDANCY,OTHERS,2023-03-15 09:00:00,Service restored,BTS,1,NOC_TEAM,NaN,NOC-RAN,0.0000,0.0000,3.3,2023-03-15 05:39:37,GMA_CORE,Fault in LAS PINAS CITY (ZONE 6) – FOC CUT WIT...
1,TKT00012789,Region 3,ZONE 3,MARIKINA CITY,Z3MAR0004,RESOLVED,3.0,1,33.2899,33.2899,FACILITIES-Cooling System Problem,FACILITIES-Cooling System Problem,Resolved – FACILITIES-Cooling System Problem,FACILITIES,2023-03-15 09:00:00,Service restored,ROUTER,0,FIELD_TEAM,FIELD_LEAD,FO_GMA,32.3451,0.9448,3.1,2023-03-15 06:17:29,GMA_CORE,Fault in MARIKINA CITY (ZONE 3) – FACILITIES-C...
2,TKT00017501,Region 3,ZONE 4,MANILA,Z4MAN0010,RESOLVED,3.0,1,37.0024,37.0024,EQUIPMENT-Defective Hardware,EQUIPMENT-Defective Hardware,Resolved – EQUIPMENT-Defective Hardware,EQUIPMENT,2023-03-15 09:00:00,Service restored,BTS,1,NOC_TEAM,NaN,NOC-RAN,0.0000,0.0000,3.1,2023-03-15 08:07:02,GMA_CORE,Fault in MANILA (ZONE 4) – EQUIPMENT-Defective...
3,TKT00034762,Region 3,ZONE 6,PARANAQUE CITY,Z6PAR0020,RESOLVED,3.0,1,99.6935,99.6935,FOC CUT - LINEAR,single fiber cut,Resolved – FOC CUT - LINEAR,OTHERS,2023-03-15 09:00:00,Service restored,BTS,0,NOC_TEAM,NaN,NOC-RAN,0.0000,0.0000,3.1,2023-03-15 05:57:19,GMA_CORE,Fault in PARANAQUE CITY (ZONE 6) – FOC CUT - L...
4,TKT00035101,Region 3,ZONE 6,MUNTINLUPA CITY,Z6MUN0032,RESOLVED,3.0,2,95.2999,95.2999,FOC CUT - LINEAR,fiber optic cut,Resolved – FOC CUT - LINEAR,OTHERS,2023-03-15 09:00:00,Service restored,BTS,0,FIELD_TEAM,FIELD_LEAD,FO_GMA,94.4276,0.8723,3.2,2023-03-15 08:25:03,GMA_CORE,Fault in MUNTINLUPA CITY (ZONE 6) – FOC CUT - ...


## 3. Schema & Data Types

In [3]:
print("📊 Data Types per Column:")
print(df_raw.dtypes)

📊 Data Types per Column:
TICKETID                  object
Region                    object
ZONE                      object
CITY                      object
Area                      object
FT Status                 object
Priority                 float64
Urgency                    int64
OUTAGEDURATION           float64
RESOLUTION_TIME_HOURS    float64
Standardized RFO          object
RFODescription            object
RootCause                 object
Fault Type                object
StartDateTime             object
ActionTaken               object
NEType                    object
SLA_Compliant              int64
FT_Owner                  object
WOLeadName                object
WOOwnerGroup              object
FIELD_TIME_HOURS         float64
DISPATCH_DELAY_HOURS     float64
Priority_Urgency         float64
REPORTDATE                object
ContactGroup              object
DESCRIPTION               object
dtype: object


## 4. Missing Values

In [4]:
print("\n⚠️ Missing Values:")
missing = df_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print(missing)
else:
    print("No missing values detected!")


⚠️ Missing Values:
WOLeadName        14800
WOOwnerGroup       2408
FT_Owner           2246
NEType              629
OUTAGEDURATION      150
Priority             80
dtype: int64


## 5. Column Distributions

In [5]:
# Numeric summary
print("\n🔢 Numeric Columns Summary:")
print(df_raw.describe().T)

# Categorical summary
categorical_cols = df_raw.select_dtypes(include='object').columns
for col in categorical_cols:
    print(f"\n📋 {col} value counts:")
    print(df_raw[col].value_counts())


🔢 Numeric Columns Summary:
                         count       mean        std     min      25%  \
Priority               38239.0   2.912210   0.352981  1.0000   3.0000   
Urgency                38319.0   2.160730   1.808539 -1.0000   1.0000   
OUTAGEDURATION         38169.0  58.773118  26.115307  0.1100  40.5698   
RESOLUTION_TIME_HOURS  38319.0  57.979476  23.595130  0.1999  40.6465   
SLA_Compliant          38319.0   0.821812   0.382676  0.0000   1.0000   
FIELD_TIME_HOURS       38319.0  34.264083  32.606673 -0.8956   0.0000   
DISPATCH_DELAY_HOURS   38319.0   0.496402   0.503028  0.0000   0.0000   
Priority_Urgency       38319.0   3.106007   0.321653  1.1000   3.1000   

                           50%       75%       max  
Priority                3.0000   3.00000    4.0000  
Urgency                 2.0000   3.00000   99.0000  
OUTAGEDURATION         55.5305  73.25340  199.7000  
RESOLUTION_TIME_HOURS  55.2587  72.54100  170.6841  
SLA_Compliant           1.0000   1.00000    1.000

## 6. Outlier Checks

In [6]:
# Example safe check for outliers
if 'OUTAGEDURATION' in df_raw.columns:
    threshold = df_raw['OUTAGEDURATION'].quantile(0.99)
    outliers_raw = df_raw[df_raw['OUTAGEDURATION'] > threshold]
    display(outliers_raw[['TICKETID', 'OUTAGEDURATION']] if 'TICKETID' in df_raw.columns else outliers_raw)

,TICKETID,OUTAGEDURATION
58,TKT00037548,191.0300
128,TKT00037476,192.9400
132,TKT00037167,165.5800
153,TKT00024974,143.2066
337,TKT00037469,163.3700
...,...,...
37854,TKT00023986,137.9134
37970,TKT00037647,145.5800
38093,TKT00037079,191.1600
38098,TKT00029038,139.6533


## 7. Data Quality Summary Table

In [7]:
summary_table = pd.DataFrame({
    'Column': df_raw.columns,
    'DataType': df_raw.dtypes,
    'MissingCount': df_raw.isnull().sum(),
    'MissingPercent': df_raw.isnull().mean() * 100,
    'UniqueValues': df_raw.nunique()
})

summary_table

,Column,DataType,MissingCount,MissingPercent,UniqueValues
TICKETID,TICKETID,object,0,0.000000,38257
Region,Region,object,0,0.000000,1
ZONE,ZONE,object,0,0.000000,6
CITY,CITY,object,0,0.000000,19
Area,Area,object,0,0.000000,597
FT Status,FT Status,object,0,0.000000,4
Priority,Priority,float64,80,0.208774,4
Urgency,Urgency,int64,0,0.000000,6
OUTAGEDURATION,OUTAGEDURATION,float64,150,0.391451,37039
RESOLUTION_TIME_HOURS,RESOLUTION_TIME_HOURS,float64,0,0.000000,36081


## 8. Commentary

### 📌 Observations:
- Most columns are complete; only a few have missing values (list from summary table)
- `OUTAGEDURATION` contains extreme outliers; may need capping or filtering for analysis
- Categorical columns appear consistent; some categories may need standardization
- Dates and IDs appear valid

✅ Overall, the dataset is suitable for Project 1 analysis, but keep in mind missing values and outliers during cleaning.